In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

import warnings 
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns


sns.set(style='white',context='notebook',palette='muted')
import matplotlib.pyplot as plt

train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')
test = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')
display(train.head())

In [ ]:
print('Train dataset size:',train.shape)
print('Test dataset size:',test.shape)

In [ ]:
display(train.head(),test.head())

In [ ]:

full = pd.concat([train,test],ignore_index=True)
full.describe()

In [ ]:
full.info()

In [ ]:
full_data = full.drop(['Id'],axis=1)

In [ ]:

fullSel = pd.get_dummies(full_data)
fullSel.head()

In [ ]:
experData=fullSel[fullSel['Cover_Type'].notnull()] 
preData=fullSel[fullSel['Cover_Type'].isnull()] 

experData_X=experData.drop('Cover_Type',axis=1)
experData_y=experData['Cover_Type']

preData_X=preData.drop('Cover_Type',axis=1)


from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,ExtraTreesClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV,cross_val_score,StratifiedKFold


kfold=StratifiedKFold(n_splits=10)


classifiers=[]
classifiers.append(SVC())
classifiers.append(DecisionTreeClassifier())
classifiers.append(RandomForestClassifier())
classifiers.append(ExtraTreesClassifier())
classifiers.append(GradientBoostingClassifier())
classifiers.append(KNeighborsClassifier())
classifiers.append(LogisticRegression())
classifiers.append(LinearDiscriminantAnalysis())

In [ ]:

cv_results = []
for classifier in classifiers:
    cv_results.append(cross_val_score(classifier, experData_X, experData_y,
                                      scoring='accuracy', cv=kfold, n_jobs=-1))


cv_means = []
cv_std = []
for cv_result in cv_results:
    cv_means.append(cv_result.mean())
    cv_std.append(cv_result.std())


cvResDf = pd.DataFrame({'cv_mean': cv_means,
                        'cv_std': cv_std,
                        'algorithm': ['SVC', 'DecisionTreeCla', 'RandomForestCla', 'ExtraTreesCla',
                                      'GradientBoostingCla', 'KNN', 'LR', 'LDA']})

cvResDf

In [ ]:
bar = sns.barplot(data=cvResDf.sort_values(by='cv_mean',ascending=False),
                  x='cv_mean',y='algorithm',**{'xerr':cv_std})
bar.set(xlim = (0.2,0.9))

In [ ]:
#SVC improvement
svc = SVC()
gb_param_grid = {'C' : [0.1,0.5,1,2,3,5,10],
              'kernel':['rbf','poly','sigmoid']
              }
modelgsSVC = GridSearchCV(svc,param_grid = gb_param_grid, cv=kfold, 
                                     scoring="accuracy", n_jobs= -1, verbose = 1)
modelgsSVC.fit(experData_X,experData_y)
modelgsSVC.best_score_

In [ ]:
# Ramdom forest improvement
rf = RandomForestClassifier(random_state=42)

rf_param_grid = {
    'n_estimators': [100, 200, 300, 1000],  
    'max_depth': [None, 10, 20, 30, 50],  
    'min_samples_split': [2, 5, 10, 20], 
    'min_samples_leaf': [1, 2, 4, 10]     
}
modelgsRF = GridSearchCV(estimator=rf, param_grid=rf_param_grid, cv=3, n_jobs=-1, verbose=2, scoring='accuracy')
modelgsRF.fit(experData_X,experData_y)
modelgsRF.best_score_

In [ ]:
# extra tree improvement
ext = ExtraTreesClassifier(random_state=42)


ext_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

modelgsEXT= GridSearchCV(estimator=ext, param_grid=ext_param_grid, cv=3, n_jobs=-1, verbose=2, scoring='accuracy')
modelgsEXT.fit(experData_X,experData_y)
modelgsEXT.best_score_

In [ ]:
y_ = modelgsRF.predict(preData_X)
y_ = y_.astype(int)

In [ ]:
RFpreResultDf=pd.DataFrame()
RFpreResultDf['Id']=full['Id'][full['Cover_Type'].isnull()]
RFpreResultDf['Cover_Type']= y_
RFpreResultDf

In [ ]:
RFpreResultDf.to_csv('Forest_submission.csv',index=False)